In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 727.4 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 69.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [3]:
import dagshub
import mlflow
import mlflow.sklearn

#dagshub.init(repo_owner='YOUR_DAGSHUB_USERNAME', repo_name='YOUR_REPO_NAME', mlflow=True)
dagshub.init(repo_owner='mkhak23', repo_name='ML_assignment2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=788c6870-90ad-41d3-b803-b9df39f9dda1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c1818490a4717cb9ab009cf0ea69ca17c6e9b4db533c667dc41682d403d0c3e0




Accessing as mkhak23

Initialized MLflow to track repo "mkhak23/ML_assignment2"

Repository mkhak23/ML_assignment2 initialized!

In [4]:
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')

# **Data Cleaning**

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class TransactionCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        transaction = X["transaction"]

        nan_ratio = transaction.isna().mean()
        self.cols_to_drop_ = nan_ratio[nan_ratio > self.threshold].index.tolist()

        return self

    def transform(self, X):
        transaction = X["transaction"].copy()
        identity = X["identity"].copy()

        transaction = transaction.drop(columns=self.cols_to_drop_, errors="ignore")

        merged = transaction.merge(identity, on="TransactionID", how="left")

        return merged

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

class LogisticImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.num_imputer = SimpleImputer(strategy="median")
        self.cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")

    def fit(self, X, y=None):
        X = X.copy()

        self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = X.select_dtypes(include=["object"]).columns.tolist()

        self.num_imputer.fit(X[self.numeric_cols_])

        if len(self.categorical_cols_) > 0:
            self.cat_imputer.fit(X[self.categorical_cols_])

        return self

    def transform(self, X):
        X = X.copy()

        X[self.numeric_cols_] = self.num_imputer.transform(X[self.numeric_cols_])

        if len(self.categorical_cols_) > 0:
            X[self.categorical_cols_] = self.cat_imputer.transform(X[self.categorical_cols_])

        return X

# **Ordinal Encoding**

In [7]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

class CategoricalOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

    def fit(self, X, y=None):
        X = X.copy()

        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        else:
            self.cols_ = self.cols

        X_cat = X[self.cols_].fillna("missing")

        self.encoder.fit(X_cat)

        return self

    def transform(self, X):
        X = X.copy()

        X_cat = X[self.cols_].fillna("missing")

        encoded = self.encoder.transform(X_cat)

        encoded_df = pd.DataFrame(
            encoded,
            columns=self.cols_,
            index=X.index
        )

        X = X.drop(columns=self.cols_)
        X = pd.concat([X, encoded_df], axis=1)

        return X

# **Feature Engineering**

In [8]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, uid_cols=("card1", "addr1")):
        self.uid_cols = uid_cols
        self.uid_means_ = None

    def fit(self, X, y=None):
        X = X.copy()

        uid = self._make_uid(X)

        self.uid_means_ = (
            pd.DataFrame({
                "uid": uid,
                "TransactionAmt": X["TransactionAmt"]
            })
            .groupby("uid")["TransactionAmt"]
            .mean()
        )

        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = (X["TransactionDT"] // 3600) % 24

        id_cols = [col for col in X.columns if col.startswith("id_")]
        if id_cols:
            X["identity_missing"] = X[id_cols].isna().sum(axis=1)
        else:
            X["identity_missing"] = 0

        uid = self._make_uid(X)

        uid_mean = uid.map(self.uid_means_)

        global_mean = self.uid_means_.mean()
        uid_mean = uid_mean.fillna(global_mean)

        X["uid_amt_mean"] = uid_mean

        X["uid_amt_diff"] = X["TransactionAmt"] - X["uid_amt_mean"]

        return X

    def _make_uid(self, X):
        uid = X[self.uid_cols[0]].astype(str)
        for col in self.uid_cols[1:]:
            uid += "_" + X[col].astype(str)
        return uid

# **Feature Selection**

In [9]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class UselessFeatureDropper(BaseEstimator, TransformerMixin):
    def __init__(self,  variance_threshold=0.0):
        self.variance_threshold = variance_threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        X = X.copy()
        self.cols_to_drop_ = []

        constant_cols = [
            col for col in X.columns
            if X[col].nunique(dropna=False) <= 1
        ]

        numeric_cols = X.select_dtypes(include=["number"]).columns

        near_zero_var_cols = [
            col for col in numeric_cols
            if X[col].var(skipna=True) <= self.variance_threshold
        ]

        self.cols_to_drop_ = list(set(
            constant_cols + near_zero_var_cols
        ))

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [11]:
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

mlflow.set_experiment("random_forest_training")

train_trx, val_trx = train_test_split(
    transaction,
    test_size=0.2,
    random_state=42,
    stratify=transaction["isFraud"]
)



y_train = train_trx["isFraud"]
y_val = val_trx["isFraud"]
train_trx = train_trx.drop(columns=["isFraud"])
val_trx = val_trx.drop(columns=["isFraud"])

X_train_raw = {
    "transaction": train_trx,
    "identity": identity
}

X_val_raw = {
    "transaction": val_trx,
    "identity": identity
}

cleaning_fe_pipeline = Pipeline([
    ("transaction_cleaner", TransactionCleaner()),
    ("drop_useless", UselessFeatureDropper()),
    ("features", FraudFeatureEngineer()),
    ("imputer", LogisticImputer()),
    ("cat_ordinal_encoding", CategoricalOrdinalEncoder())
])

with mlflow.start_run(run_name="cleaning_feature_engineering"):

    X_train_processed = cleaning_fe_pipeline.fit_transform(X_train_raw, y_train)
    X_val_processed = cleaning_fe_pipeline.transform(X_val_raw)

    mlflow.log_param("transaction_cleaner", True)
    mlflow.log_param("drop_useless", True)
    mlflow.log_param("feature_engineering", True)
    mlflow.log_param("imputer", "enabled")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("max_categories", 50)

    mlflow.log_param("train_rows_before", train_trx.shape[0])
    mlflow.log_param("train_transaction_cols_before", train_trx.shape[1])
    mlflow.log_param("val_rows_before", val_trx.shape[0])
    mlflow.log_param("val_transaction_cols_before", val_trx.shape[1])
    mlflow.log_param("identity_cols_before", identity.shape[1])

    mlflow.log_param("train_rows_after", X_train_processed.shape[0])
    mlflow.log_param("train_cols_after", X_train_processed.shape[1])
    mlflow.log_param("val_rows_after", X_val_processed.shape[0])
    mlflow.log_param("val_cols_after", X_val_processed.shape[1])

    dropped_cols = cleaning_fe_pipeline.named_steps["drop_useless"].cols_to_drop_

    mlflow.log_param("num_useless_dropped_cols", len(dropped_cols))

🏃 View run cleaning_feature_engineering at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4/runs/558ab2e638674f88b478250aae8856a3
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4


# **Random Forest**

In [12]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    log_loss
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import mlflow

mlflow.set_experiment("random_forest_training")

param_list = [
    {"n_estimators": 100, "max_depth": 8,  "min_samples_leaf": 50},
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 50},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 100},
]

results = []

for params in param_list:
    with mlflow.start_run(run_name="RandomForest_training"):

        model = RandomForestClassifier(
            **params,
            class_weight="balanced",
            random_state=42
        )

        temp_pipeline = Pipeline([
            ("transaction_cleaner", TransactionCleaner()),
            ("drop_useless", UselessFeatureDropper()),
            ("features", FraudFeatureEngineer()),
            ("imputer", LogisticImputer()),
            ("cat_enc", CategoricalOrdinalEncoder()),
            ("model", model)
        ])

        temp_pipeline.fit(X_train_raw, y_train)

        train_preds = temp_pipeline.predict_proba(X_train_raw)[:, 1]
        val_preds = temp_pipeline.predict_proba(X_val_raw)[:, 1]

        train_auc = roc_auc_score(y_train, train_preds)
        val_auc = roc_auc_score(y_val, val_preds)
        auc_gap = train_auc - val_auc

        y_val_pred = (val_preds >= 0.5).astype(int)

        pr_auc = average_precision_score(y_val, val_preds)
        precision = precision_score(y_val, y_val_pred, zero_division=0)
        recall = recall_score(y_val, y_val_pred, zero_division=0)
        f1 = f1_score(y_val, y_val_pred, zero_division=0)
        ll = log_loss(y_val, val_preds)

        mlflow.log_params(params)

        mlflow.log_metric("train_roc_auc", train_auc)
        mlflow.log_metric("val_roc_auc", val_auc)
        mlflow.log_metric("auc_gap", auc_gap)
        mlflow.log_metric("val_pr_auc", pr_auc)
        mlflow.log_metric("val_precision", precision)
        mlflow.log_metric("val_recall", recall)
        mlflow.log_metric("val_f1", f1)
        mlflow.log_metric("val_log_loss", ll)

        results.append({
            **params,
            "train_roc_auc": train_auc,
            "val_roc_auc": val_auc,
            "auc_gap": auc_gap,
            "val_pr_auc": pr_auc,
            "val_precision": precision,
            "val_recall": recall,
            "val_f1": f1,
            "val_log_loss": ll
        })

        print(params)
        print("Train AUC:", train_auc)
        print("Val AUC:", val_auc)
        print("Gap:", auc_gap)
        print("-" * 40)

results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False)
results_df

{'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 50}
Train AUC: 0.8784654197010208
Val AUC: 0.87321678981923
Gap: 0.005248629881790756
----------------------------------------
🏃 View run RandomForest_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4/runs/c07ce7329b604dfda3bb03a7b5db5234
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4
{'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 50}
Train AUC: 0.8949076443533546
Val AUC: 0.8850970465621932
Gap: 0.009810597791161335
----------------------------------------
🏃 View run RandomForest_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4/runs/9fcdf26b075247ff8e736b30aedebec5
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/4
{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 100}
Train AUC: 0.9048653319119115
Val AUC: 0.8924074617668383
Gap: 0.012457870145073269
---------------------

,n_estimators,max_depth,min_samples_leaf,train_roc_auc,val_roc_auc,auc_gap,val_pr_auc,val_precision,val_recall,val_f1,val_log_loss
2,300,12,100,0.904865,0.892407,0.012458,0.523900,0.178641,0.755625,0.288966,0.377053
1,200,10,50,0.894908,0.885097,0.009811,0.513781,0.170772,0.751512,0.278303,0.392858
0,100,8,50,0.878465,0.873217,0.005249,0.470552,0.159333,0.744737,0.262505,0.423818


In [13]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# get best params from your results_df
best_params = results_df.iloc[0][[
    "n_estimators",
    "max_depth",
    "min_samples_leaf"
]].to_dict()

best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])

final_model = RandomForestClassifier(
    **best_params,
    class_weight="balanced",
    random_state=42
)

final_pipeline = Pipeline([
    ("transaction_cleaner", TransactionCleaner()),
    ("drop_useless", UselessFeatureDropper()),
    ("features", FraudFeatureEngineer()),
    ("imputer", LogisticImputer()),
    ("cat_enc", CategoricalOrdinalEncoder()),
    ("model", final_model)
])

# train on full raw train data
X_full_raw = {
    "transaction": transaction.drop(columns=["isFraud"]),
    "identity": identity
}

y_full = transaction["isFraud"]

final_pipeline.fit(X_full_raw, y_full)

mlflow.set_experiment("random_forest")

with mlflow.start_run(run_name="final_best_random_forest"):
    mlflow.log_params(best_params)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("encoder", "OrdinalEncoder")
    mlflow.log_param("imputer", "median_numeric_missing_categorical")
    mlflow.log_param("n_jobs", 2)

    mlflow.log_metric("best_val_roc_auc", results_df.iloc[0]["val_roc_auc"])
    mlflow.log_metric("best_val_pr_auc", results_df.iloc[0]["val_pr_auc"])
    mlflow.log_metric("best_val_f1", results_df.iloc[0]["val_f1"])

    mlflow.sklearn.log_model(
        final_pipeline,
        name="final_pipeline"
    )

2026/05/05 20:24:18 INFO mlflow.tracking.fluent: Experiment with name 'random_forest' does not exist. Creating a new experiment.
2026/05/05 20:24:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_best_random_forest at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/5/runs/4908000d736e428684b9e0d16ef97c9d
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/5
